In [ ]:
# In[1]:

import pandas as pd

# Load telemetry files (they are in current working directory)
# Reuse variable names for efficiency
df_metric = pd.read_csv("metric.csv")
df_log = pd.read_csv("log.csv")
df_trace = pd.read_csv("trace.csv")
df_error = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (Unix seconds -> UTC) per rules
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# Helper to truncate lists to max_n items
def _truncate_list(x, max_n=50):
    if isinstance(x, list):
        return x[:max_n]
    else:
        # convert any iterable to list and truncate
        return list(x)[:max_n]

# 1) Inventory from metric.csv
# For each cmdb_id: cmdb_id, total_point_count, unique_kpi_count, list_of_kpi_names (up to 50)
metrics_inv = (
    df_metric
    .groupby('cmdb_id', sort=False)
    .agg(
        total_point_count=('kpi_name', 'size'),
        unique_kpi_count=('kpi_name', 'nunique'),
        list_of_kpi_names=('kpi_name', lambda s: _truncate_list(list(pd.unique(s)), 50))
    )
    .reset_index()
)

# 2) Inventory from log.csv
logs_inv = (
    df_log
    .groupby('cmdb_id', sort=False)
    .agg(
        total_point_count=('log_name', 'size'),
        unique_log_name_count=('log_name', 'nunique'),
        list_of_log_names=('log_name', lambda s: _truncate_list(list(pd.unique(s)), 50))
    )
    .reset_index()
)

# 3) Inventory from trace.csv
traces_inv = (
    df_trace
    .groupby('cmdb_id', sort=False)
    .agg(
        total_point_count=('trace_name', 'size'),
        unique_trace_name_count=('trace_name', 'nunique'),
        list_of_trace_names=('trace_name', lambda s: _truncate_list(list(pd.unique(s)), 50))
    )
    .reset_index()
)

# 4) Top 20 rows from error_logs.csv with selected columns
# Keep timestamp (datetime UTC), cmdb_id, message
error_top = df_error[['timestamp', 'cmdb_id', 'message']].head(20).reset_index(drop=True)

# Display compact inventories (variables returned)
metrics_inv, logs_inv, traces_inv, error_top

```
Out[1]:
```
```python
# Compose a plain-English compact summary using existing variables from prior execution:
metric_ids = list(metrics_inv['cmdb_id']) if 'metrics_inv' in globals() else []
log_ids = list(logs_inv['cmdb_id']) if 'logs_inv' in globals() else []
trace_empty = traces_inv.empty if 'traces_inv' in globals() else True
error_empty = error_top.empty if 'error_top' in globals() else True

# Build short examples for metrics (top by total_point_count)
if metric_ids:
    examples = metrics_inv.sort_values('total_point_count', ascending=False).head(3)
    metric_examples = "; ".join(
        f"{row.cmdb_id} ({int(row.total_point_count)} pts, {int(row.unique_kpi_count)} KPIs)"
        for _, row in examples.iterrows()
    )
else:
    metric_examples = "no metric entries"

# Build log summary (counts range and common log names)
if log_ids:
    log_counts = logs_inv['total_point_count'].agg(['min','max'])
    common_log_names = []
    if 'list_of_log_names' in logs_inv.columns:
        uniq_names = sorted({name for lst in logs_inv['list_of_log_names'] for name in lst})
        common_log_names = uniq_names[:50]
    log_summary = f"counts range {int(log_counts['min'])}–{int(log_counts['max'])}; log names: {common_log_names}"
else:
    log_summary = "no log entries"

summary = (
    "Telemetry inventory summary:\n"
    f"- metric.csv: {len(metric_ids)} cmdb_id entries. Examples: {metric_examples}. "
    "Many services include KPIs like cpu, latency-50, latency-90, mem, socket; DB nodes (suffix -db) typically have 100 pts and 4 KPIs (cpu,diskio,mem,socket).\n"
    f"- log.csv: {len(log_ids)} cmdb_id entries; {log_summary}.\n"
    f"- trace.csv: {'no trace data (empty)' if trace_empty else 'trace data present'}.\n"
    f"- error_logs.csv: {'no error log rows (empty)' if error_empty else f'top {min(20, len(error_top))} error rows available'}.\n"
    "Lists truncated to max 50 items where applicable."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id  total_point_count  unique_kpi_count                                  list_of_kpi_names
0               carts                150                 6  [cpu, latency-50, latency-90, mem, socket, wor...
1            carts-db                100                 4                         [cpu, diskio, mem, socket]
2           catalogue                150                 6  [cpu, latency-50, latency-90, mem, socket, wor...
3        catalogue-db                100                 4                         [cpu, diskio, mem, socket]
4           front-end                161                 7  [cpu, latency-50, latency-90, mem, socket, wor...
5              orders                175                 7  [cpu, error, latency-50, latency-90, mem, sock...
6           orders-db                100                 4                         [cpu, diskio, mem, socket]
7             payment                159                 7  [cpu, diskio, latency-50, latency-90, mem, soc...
8        queue-master                100                 4                         [cpu, diskio, mem, socket]
9            rabbitmq                100                 4                         [cpu, diskio, mem, socket]
10  rabbitmq-exporter                 75                 3                                 [cpu, mem, socket]
11         session-db                100                 4                         [cpu, diskio, mem, socket]
12           shipping                144                 6  [cpu, latency-50, latency-90, mem, socket, wor...
13               user                150                 6  [cpu, latency-50, latency-90, mem, socket, wor...
14            user-db                100                 4                         [cpu, diskio, mem, socket],          cmdb_id  total_point_count  unique_log_name_count                 list_of_log_names
0       carts-db                 48                      2  [log.error_count, log.row_count]
1      catalogue                 50                      2  [log.error_count, log.row_count]
2      front-end                 50                      2  [log.error_count, log.row_count]
3         orders                 50                      2  [log.error_count, log.row_count]
4        payment                 42                      2  [log.error_count, log.row_count]
5   queue-master                 42                      2  [log.error_count, log.row_count]
6       shipping                 42                      2  [log.error_count, log.row_count]
7           user                 50                      2  [log.error_count, log.row_count]
8          carts                 44                      2  [log.error_count, log.row_count]
9     session-db                 10                      2  [log.error_count, log.row_count]
10       user-db                  2                      2  [log.error_count, log.row_count], Empty DataFrame
Columns: [cmdb_id, total_point_count, unique_trace_name_count, list_of_trace_names]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing dataframes if present, otherwise load and parse timestamps per rules
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
try:
    df_error
except NameError:
    df_error = pd.read_csv("error_logs.csv")

# Ensure timestamps are parsed as UTC datetimes (Unix seconds)
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start = pd.Timestamp("2024-01-20 17:50:28", tz="UTC")
end = pd.Timestamp("2024-01-20 18:20:28", tz="UTC")

# --------------------------
# 1) Metrics: global P95 and window breaches
# --------------------------
# Global stats per (cmdb_id, kpi_name)
metrics_g = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], sort=False)
    .agg(
        total_point_count=('value', 'size'),
        p95_threshold=('value', lambda s: float(s.quantile(0.95)))
    )
    .reset_index()
)

# Windowed metric rows
df_metric_win = df_metric[(df_metric['timestamp'] >= start) & (df_metric['timestamp'] <= end)].copy()

# Attach p95_threshold to window rows for breach evaluation
if not df_metric_win.empty:
    df_metric_win = df_metric_win.merge(metrics_g[['cmdb_id','kpi_name','p95_threshold']], on=['cmdb_id','kpi_name'], how='left')
    df_metric_win['is_breach'] = df_metric_win['value'] > df_metric_win['p95_threshold']
    df_metric_win['breach_ts'] = df_metric_win['timestamp'].where(df_metric_win['is_breach'], pd.NaT)

    window_stats = (
        df_metric_win
        .groupby(['cmdb_id','kpi_name'], sort=False)
        .agg(
            points_in_window=('value','size'),
            breach_count=('is_breach','sum'),
            earliest_breach_timestamp=('breach_ts','min'),
            max_value_in_window=('value','max')
        )
        .reset_index()
    )
    # convert earliest_breach_timestamp to unix seconds (int) or NaN
    def ts_to_unix(t):
        if pd.isna(t):
            return np.nan
        return int(t.timestamp())
    window_stats['earliest_breach_timestamp'] = window_stats['earliest_breach_timestamp'].apply(ts_to_unix)
else:
    # empty window stats
    window_stats = pd.DataFrame(columns=['cmdb_id','kpi_name','points_in_window','breach_count','earliest_breach_timestamp','max_value_in_window'])

# Merge global metrics with window stats to get full view; fill zeros where appropriate
metric_full = metrics_g.merge(window_stats, on=['cmdb_id','kpi_name'], how='left')
metric_full['points_in_window'] = metric_full['points_in_window'].fillna(0).astype(int)
metric_full['breach_count'] = metric_full['breach_count'].fillna(0).astype(int)
metric_full['max_value_in_window'] = metric_full['max_value_in_window'].astype(float)
metric_full['earliest_breach_timestamp'] = metric_full['earliest_breach_timestamp'].astype(float)  # ints where present, NaN otherwise

# Compute breach_ratio per instructions: (max_value_in_window - p95)/p95 if points exist; else NaN. Avoid division by zero.
def compute_ratio(row):
    if row['points_in_window'] == 0 or pd.isna(row['max_value_in_window']) or pd.isna(row['p95_threshold']):
        return np.nan
    if row['p95_threshold'] == 0:
        return np.nan
    return (row['max_value_in_window'] - row['p95_threshold']) / row['p95_threshold']

metric_full['breach_ratio'] = metric_full.apply(compute_ratio, axis=1)

# Select and order columns required
metric_result = metric_full[[
    'cmdb_id','kpi_name','total_point_count','p95_threshold',
    'points_in_window','breach_count','earliest_breach_timestamp',
    'max_value_in_window','breach_ratio'
]]

# Top 20 rows sorted by breach_count desc then breach_ratio desc
metric_kpi_breaches_top20 = metric_result.sort_values(['breach_count','breach_ratio'], ascending=[False, False]).head(20).reset_index(drop=True)

# Aggregated per-cmdb_id sum of breach_counts across all KPIs, sorted desc (top 20)
breaches_per_cmdb = (
    metric_kpi_breaches_top20[['cmdb_id','breach_count']]
    .groupby('cmdb_id', sort=False)
    .agg(sum_of_breach_counts=('breach_count','sum'))
    .reset_index()
    .sort_values('sum_of_breach_counts', ascending=False)
)
# We should compute aggregated across ALL KPIs, not just top20; recalc across full metric_result:
breaches_per_cmdb_all = (
    metric_result
    .groupby('cmdb_id', sort=False)
    .agg(sum_of_breach_counts=('breach_count','sum'))
    .reset_index()
    .sort_values('sum_of_breach_counts', ascending=False)
)
breaches_per_cmdb_top20 = breaches_per_cmdb_all.head(20).reset_index(drop=True)

# --------------------------
# 2) Logs: global P95 and window breaches
# --------------------------
logs_g = (
    df_log
    .groupby(['cmdb_id','log_name'], sort=False)
    .agg(
        total_point_count=('value','size'),
        p95_threshold=('value', lambda s: float(s.quantile(0.95)))
    )
    .reset_index()
)

df_log_win = df_log[(df_log['timestamp'] >= start) & (df_log['timestamp'] <= end)].copy()

if not df_log_win.empty:
    df_log_win = df_log_win.merge(logs_g[['cmdb_id','log_name','p95_threshold']], on=['cmdb_id','log_name'], how='left')
    df_log_win['is_breach'] = df_log_win['value'] > df_log_win['p95_threshold']
    df_log_win['breach_ts'] = df_log_win['timestamp'].where(df_log_win['is_breach'], pd.NaT)

    log_window_stats = (
        df_log_win
        .groupby(['cmdb_id','log_name'], sort=False)
        .agg(
            points_in_window=('value','size'),
            breach_count=('is_breach','sum'),
            earliest_breach_timestamp=('breach_ts','min'),
            max_value_in_window=('value','max')
        )
        .reset_index()
    )
    log_window_stats['earliest_breach_timestamp'] = log_window_stats['earliest_breach_timestamp'].apply(lambda t: int(t.timestamp()) if pd.notna(t) else np.nan)
else:
    log_window_stats = pd.DataFrame(columns=['cmdb_id','log_name','points_in_window','breach_count','earliest_breach_timestamp','max_value_in_window'])

log_full = logs_g.merge(log_window_stats, on=['cmdb_id','log_name'], how='left')
log_full['points_in_window'] = log_full['points_in_window'].fillna(0).astype(int)
log_full['breach_count'] = log_full['breach_count'].fillna(0).astype(int)
log_full['max_value_in_window'] = log_full['max_value_in_window'].astype(float)
log_full['earliest_breach_timestamp'] = log_full['earliest_breach_timestamp'].astype(float)

# Keep only rows where breach_count > 0
log_breaches = log_full[log_full['breach_count'] > 0].copy()
log_breaches = log_breaches[[
    'cmdb_id','log_name','total_point_count','p95_threshold',
    'points_in_window','breach_count','earliest_breach_timestamp','max_value_in_window'
]]
log_breaches_top20 = log_breaches.sort_values(['breach_count','max_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

# --------------------------
# 3) Note on trace.csv and error_logs.csv emptiness
# --------------------------
trace_empty = df_trace.empty
error_empty = df_error.empty
trace_error_note = f"trace.csv is {'empty' if trace_empty else 'not empty'}; error_logs.csv is {'empty' if error_empty else 'not empty'}."

# Final compact outputs (tabular). Variables returned:
metric_kpi_breaches_top20, breaches_per_cmdb_top20, log_breaches_top20, trace_error_note

```
Out[2]:
```
```python
# Compose a concise plain-English summary string using prior results
summary = (
    "Incident window: 2024-01-20 17:50:28 UTC — 2024-01-20 18:20:28 UTC.\n\n"
    "Metrics (top findings):\n"
    "- Multiple KPIs across services breached their global P95 during the window. "
    "Top per-KPI breaches (by breach_count and breach_ratio) include payment.cpu, carts.cpu, orders.latency-90, "
    "payment.latency-90 and queue-master.diskio. Most individual KPI breach_counts observed were 2 in this window.\n"
    "- Aggregated by service, the most affected services (sum of KPI breach counts) are: orders (14), front-end (13), "
    "carts (12), catalogue (12), payment (12), shipping (11), and user (10). This indicates 'orders' was the most impacted overall.\n\n"
    "Logs (top findings):\n"
    "- Elevated log.row_count breaches were observed for front-end, orders, user, carts and catalogue. "
    "front-end and orders had the largest log breaches (breach_count=2, max row_count > P95). Several other components had 1 or 2 breaches.\n\n"
    "Other telemetry:\n"
    f"- trace.csv is {'empty' if trace_empty else 'not empty'}; error_logs.csv is {'empty' if error_empty else 'not empty'}.\n\n"
    "Overall conclusion: the 'orders' service shows the highest aggregate metric breaches during the incident window, "
    "with front-end and carts also exhibiting notable metric and log-level breaches. Tracing and error-log files contain no rows."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  total_point_count  p95_threshold  points_in_window  breach_count  earliest_breach_timestamp  max_value_in_window  breach_ratio
0             payment         cpu                 25   1.514308e-01                25             2               1.705773e+09         2.532696e-01      0.672510
1               carts         cpu                 25   2.308913e+00                25             2               1.705774e+09         3.266830e+00      0.414878
2              orders  latency-90                 25   4.660701e-01                25             2               1.705774e+09         6.308333e-01      0.353516
3             payment  latency-90                 22   2.933564e-01                22             2               1.705774e+09         3.613839e-01      0.231894
4        queue-master      diskio                 25   9.719895e+04                25             2               1.705773e+09         1.197298e+05      0.231802
5              orders         cpu                 25   5.870807e+00                25             2               1.705773e+09         7.198180e+00      0.226097
6            shipping  latency-90                 22   4.885347e-03                22             2               1.705774e+09         5.967722e-03      0.221555
7   rabbitmq-exporter         cpu                 25   1.311604e-04                25             2               1.705774e+09         1.530234e-04      0.166689
8            shipping         cpu                 25   9.291087e-01                25             2               1.705773e+09         1.029276e+00      0.107810
9        queue-master         cpu                 25   1.330358e+00                25             2               1.705773e+09         1.431472e+00      0.076005
10           shipping  latency-50                 22   3.157622e-03                22             2               1.705774e+09         3.397259e-03      0.075892
11            payment  latency-50                 22   3.395280e-03                22             2               1.705774e+09         3.602261e-03      0.060962
12              carts      socket                 25   1.170667e+01                25             2               1.705774e+09         1.238333e+01      0.057802
13          front-end         cpu                 25   4.533785e+00                25             2               1.705773e+09         4.794758e+00      0.057562
14         session-db      diskio                 25   3.444710e+04                25             2               1.705773e+09         3.623585e+04      0.051927
15           carts-db      diskio                 25   2.379375e+06                25             2               1.705774e+09         2.500548e+06      0.050926
16             orders      socket                 25   1.888333e+01                25             2               1.705774e+09         1.980000e+01      0.048544
17          front-end  latency-50                 25   3.176699e-02                25             2               1.705774e+09         3.321019e-02      0.045431
18          catalogue         cpu                 25   1.743516e-01                25             2               1.705773e+09         1.819392e-01      0.043519
19             orders       error                 25   2.053909e+00                25             2               1.705774e+09         2.131250e+00      0.037656,               cmdb_id  sum_of_breach_counts
0              orders                    14
1           front-end                    13
2               carts                    12
3           catalogue                    12
4             payment                    12
5            shipping                    11
6                user                    10
7             user-db                     6
8           orders-db                     6
9            carts-db                     6
10         session-db                     6
11       queue-master                     5
12       catalogue-db                     4
13           rabbitmq                     4
14  rabbitmq-exporter                     2,         cmdb_id       log_name  total_point_count  p95_threshold  points_in_window  breach_count  earliest_breach_timestamp  max_value_in_window
0     front-end  log.row_count                 25        2051.20                25             2               1.705773e+09               2056.0
1        orders  log.row_count                 25        1806.00                25             2               1.705774e+09               1890.0
2          user  log.row_count                 25         625.80                25             2               1.705774e+09                631.0
3         carts  log.row_count                 22         173.75                22             2               1.705774e+09                175.0
4     catalogue  log.row_count                 25         167.60                25             2               1.705773e+09                168.0
5  queue-master  log.row_count                 21         603.00                21             1               1.705774e+09                612.0
6       payment  log.row_count                 21         105.00                21             1               1.705774e+09                108.0
7      shipping  log.row_count                 21          67.00                21             1               1.705774e+09                 68.0
8      carts-db  log.row_count                 24           1.00                24             1               1.705774e+09                  2.0, 'trace.csv is empty; error_logs.csv is empty.')```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing dataframes if present; otherwise load and parse timestamps
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")

# Ensure timestamps parsed as UTC datetimes (Unix seconds)
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start = pd.Timestamp("2024-01-20 17:50:28", tz="UTC")
end = pd.Timestamp("2024-01-20 18:20:28", tz="UTC")

# Compute global P95 per (cmdb_id, kpi_name) for metrics (full-series)
metrics_g = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], sort=False)
    .agg(
        total_point_count=('value', 'size'),
        p95_threshold=('value', lambda s: float(s.quantile(0.95)))
    )
    .reset_index()
)

# Compute global P95 per (cmdb_id, log_name) for logs (full-series)
logs_g = (
    df_log
    .groupby(['cmdb_id', 'log_name'], sort=False)
    .agg(
        total_point_count=('value', 'size'),
        p95_threshold=('value', lambda s: float(s.quantile(0.95)))
    )
    .reset_index()
)

# Services and KPIs of interest
services = ['orders', 'payment']
kpis_of_interest = ['cpu', 'latency-90', 'latency-50', 'socket', 'error']

# Helper to compute per (service,kpi) summary row
def summarize_kpi_for_service(cmdb_id, kpi_name):
    # Check presence in full-series
    mask_full = (metrics_g['cmdb_id'] == cmdb_id) & (metrics_g['kpi_name'] == kpi_name)
    present = mask_full.any()
    if not present:
        return {
            'cmdb_id': cmdb_id,
            'kpi_name': kpi_name,
            'present': False,
            'p95_threshold': np.nan,
            'total_point_count': 0,
            'breach_timestamps': [],          # list of unix seconds
            'breach_count': 0,
            'consecutive_run': False,
            'earliest_breach_timestamp': np.nan,
            'last_breach_timestamp': np.nan,
            'max_value_in_window': np.nan,
            'breach_ratio': np.nan
        }
    # Get full-series p95 and total count
    row = metrics_g[mask_full].iloc[0]
    p95 = float(row['p95_threshold'])
    total_count = int(row['total_point_count'])
    # Filter window rows for this series
    df_win = df_metric[
        (df_metric['cmdb_id'] == cmdb_id) &
        (df_metric['kpi_name'] == kpi_name) &
        (df_metric['timestamp'] >= start) &
        (df_metric['timestamp'] <= end)
    ].copy()
    if df_win.empty:
        return {
            'cmdb_id': cmdb_id,
            'kpi_name': kpi_name,
            'present': True,
            'p95_threshold': p95,
            'total_point_count': total_count,
            'breach_timestamps': [],
            'breach_count': 0,
            'consecutive_run': False,
            'earliest_breach_timestamp': np.nan,
            'last_breach_timestamp': np.nan,
            'max_value_in_window': np.nan,
            'breach_ratio': np.nan
        }
    # Determine breaches
    df_win['is_breach'] = df_win['value'] > p95
    breaches = df_win[df_win['is_breach']].copy()
    # breach timestamps as sorted unique unix seconds
    if not breaches.empty:
        breach_ts_vals = sorted(breaches['timestamp'].dt.tz_convert('UTC').view('int64')//10**9)
        breach_count = len(breach_ts_vals)
        earliest = int(breach_ts_vals[0])
        last = int(breach_ts_vals[-1])
        # Check consecutive run: if 0 or 1 breach -> True (single point considered consecutive)
        if breach_count <= 1:
            consecutive = True
        else:
            # compute diffs between successive timestamps in seconds
            diffs = np.diff(breach_ts_vals)
            # since metric points are per-minute, expect 60s gaps for consecutive
            consecutive = bool(np.all(diffs == 60))
    else:
        breach_ts_vals = []
        breach_count = 0
        earliest = np.nan
        last = np.nan
        consecutive = False
    # max_value_in_window (over all window rows)
    max_val = float(df_win['value'].max())
    # breach_ratio only meaningful if any breach (per instruction), else NaN
    if breach_count == 0 or p95 == 0:
        ratio = np.nan
    else:
        ratio = (max_val - p95) / p95
    return {
        'cmdb_id': cmdb_id,
        'kpi_name': kpi_name,
        'present': True,
        'p95_threshold': p95,
        'total_point_count': total_count,
        'breach_timestamps': breach_ts_vals,
        'breach_count': breach_count,
        'consecutive_run': consecutive,
        'earliest_breach_timestamp': earliest,
        'last_breach_timestamp': last,
        'max_value_in_window': max_val,
        'breach_ratio': ratio
    }

# Build summaries for each service and KPI
rows = []
for svc in services:
    for k in kpis_of_interest:
        rows.append(summarize_kpi_for_service(svc, k))

summary_df = pd.DataFrame(rows)

# Keep only requested columns and keep compact representation (breach_timestamps list may be short)
cols_order = [
    'cmdb_id','kpi_name','present','total_point_count','p95_threshold',
    'breach_timestamps','breach_count','consecutive_run',
    'earliest_breach_timestamp','last_breach_timestamp',
    'max_value_in_window','breach_ratio'
]
summary_df = summary_df[cols_order]

# 2) For 'orders' logs within window: rows for log_name in [log.row_count, log.error_count]
orders_log_names = ['log.row_count', 'log.error_count']
orders_logs_win = df_log[
    (df_log['cmdb_id'] == 'orders') &
    (df_log['log_name'].isin(orders_log_names)) &
    (df_log['timestamp'] >= start) &
    (df_log['timestamp'] <= end)
].copy()

# Convert timestamps to unix seconds for display
if not orders_logs_win.empty:
    orders_logs_win['timestamp'] = (orders_logs_win['timestamp'].dt.tz_convert('UTC').view('int64')//10**9).astype(int)

# Get full-series P95 for these orders log_names
orders_logs_p95 = logs_g[
    (logs_g['cmdb_id'] == 'orders') & (logs_g['log_name'].isin(orders_log_names))
][['log_name','total_point_count','p95_threshold']].reset_index(drop=True)

# Final compact outputs
orders_kpi_summary = summary_df[summary_df['cmdb_id']=='orders'].reset_index(drop=True)
payment_kpi_summary = summary_df[summary_df['cmdb_id']=='payment'].reset_index(drop=True)
orders_log_rows = orders_logs_win[['timestamp','log_name','value']].reset_index(drop=True)
orders_log_p95 = orders_logs_p95

orders_kpi_summary, payment_kpi_summary, orders_log_rows, orders_log_p95

```
Out[3]:
```
```python
# Build a concise plain-English summary from existing results (orders_kpi_summary, payment_kpi_summary, orders_log_rows, orders_log_p95)
import numpy as np

# Helper to format one KPI row
def fmt_kpi(row):
    if not row['present']:
        return f"- {row['kpi_name']}: MISSING"
    ts_list = row['breach_timestamps']
    ts_display = ts_list if isinstance(ts_list, list) else []
    earliest = int(row['earliest_breach_timestamp']) if not np.isnan(row['earliest_breach_timestamp']) else "NaN"
    last = int(row['last_breach_timestamp']) if not np.isnan(row['last_breach_timestamp']) else "NaN"
    maxv = (float(row['max_value_in_window']) if not pd.isna(row['max_value_in_window']) else "NaN")
    p95 = float(row['p95_threshold']) if not pd.isna(row['p95_threshold']) else "NaN"
    ratio = (float(row['breach_ratio']) if not pd.isna(row['breach_ratio']) else "NaN")
    return (f"- {row['kpi_name']}: breach_timestamps={ts_display}, breach_count={int(row['breach_count'])}, "
            f"consecutive_run={bool(row['consecutive_run'])}, earliest={earliest}, last={last}, "
            f"max_value_in_window={maxv}, p95_threshold={p95}, breach_ratio={ratio:.6g}" )

import pandas as pd
orders_df = orders_kpi_summary.copy()
payment_df = payment_kpi_summary.copy()

orders_lines = [f"Service: orders"]
for _, r in orders_df.iterrows():
    orders_lines.append(fmt_kpi(r))

payment_lines = [f"Service: payment"]
for _, r in payment_df.iterrows():
    payment_lines.append(fmt_kpi(r))

# Summarize orders logs in-window and full-series P95
log_p95_map = {row['log_name']: row['p95_threshold'] for _, row in orders_log_p95.iterrows()}
orders_logs_in_win = orders_log_rows.copy()
# Determine breaches for log.row_count vs its P95
log_notes = []
for logname, p95 in log_p95_map.items():
    subset = orders_logs_in_win[orders_logs_in_win['log_name'] == logname]
    if subset.empty:
        log_notes.append(f"- {logname}: no rows in window; full-series P95={p95}")
    else:
        maxv = float(subset['value'].max())
        breaches = subset[subset['value'] > p95]
        bc = int(breaches.shape[0])
        earliest = int(breaches['timestamp'].min()) if bc>0 else "NaN"
        last = int(breaches['timestamp'].max()) if bc>0 else "NaN"
        log_notes.append(f"- {logname}: full-series P95={p95}, rows_in_window={len(subset)}, breach_count={bc}, earliest={earliest}, last={last}, max_in_window={maxv}")

# Compose final summary string
summary_lines = []
summary_lines.append("Summary for services 'orders' and 'payment' during 2024-01-20 17:50:28–18:20:28 UTC:")
summary_lines.append("")
summary_lines.append("Orders KPI summary:")
summary_lines.extend(orders_lines[1:])  # skip repeated header
summary_lines.append("")
summary_lines.append("Payment KPI summary:")
summary_lines.extend(payment_lines[1:])
summary_lines.append("")
summary_lines.append("Orders logs (window) and full-series P95:")
summary_lines.extend(log_notes)
summary_lines.append("")
summary_lines.append("Key takeaways:")
summary_lines.append("- Both 'orders' and 'payment' had multiple KPI breaches in the window.")
summary_lines.append("- 'orders' shows breaches across cpu, latency-90, latency-50, socket, and error (all with breach_count=2 each). Breaches are not all consecutive (some are, some are not).")
summary_lines.append("- 'payment' shows CPU breaches with the largest breach_ratio (~0.6725), and multiple latency breaches; 'error' KPI is missing for payment.")
summary_lines.append("- Orders' log.row_count exceeded its full-series P95 (1806) in the window (max observed 1890); log.error_count has P95=0 and observed error_count values are 0 in-window.")
summary_lines.append("")
summary_lines.append("Note: timestamps shown as Unix seconds; breach_ratio is (max_in_window - p95)/p95 (NaN if no breach).")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id    kpi_name  present  total_point_count  p95_threshold         breach_timestamps  breach_count  consecutive_run  earliest_breach_timestamp  last_breach_timestamp  max_value_in_window  breach_ratio
0  orders         cpu     True                 25       5.870807  [1705773480, 1705773660]             2            False               1.705773e+09           1.705774e+09             7.198180      0.226097
1  orders  latency-90     True                 25       0.466070  [1705774320, 1705774380]             2             True               1.705774e+09           1.705774e+09             0.630833      0.353516
2  orders  latency-50     True                 25       0.042564  [1705774260, 1705774380]             2            False               1.705774e+09           1.705774e+09             0.043366      0.018825
3  orders      socket     True                 25      18.883333  [1705773600, 1705774380]             2            False               1.705774e+09           1.705774e+09            19.800000      0.048544
4  orders       error     True                 25       2.053909  [1705774140, 1705774620]             2            False               1.705774e+09           1.705775e+09             2.131250      0.037656,    cmdb_id    kpi_name  present  total_point_count  p95_threshold         breach_timestamps  breach_count  consecutive_run  earliest_breach_timestamp  last_breach_timestamp  max_value_in_window  breach_ratio
0  payment         cpu     True                 25       0.151431  [1705773180, 1705773240]             2             True               1.705773e+09           1.705773e+09             0.253270      0.672510
1  payment  latency-90     True                 22       0.293356  [1705774200, 1705774380]             2            False               1.705774e+09           1.705774e+09             0.361384      0.231894
2  payment  latency-50     True                 22       0.003395  [1705774200, 1705774380]             2            False               1.705774e+09           1.705774e+09             0.003602      0.060962
3  payment      socket     True                 25       8.000000              [1705774200]             1             True               1.705774e+09           1.705774e+09             8.733333      0.091667
4  payment       error    False                  0            NaN                        []             0            False                        NaN                    NaN                  NaN           NaN,      timestamp         log_name  value
0   1705773180  log.error_count      0
1   1705773180    log.row_count     70
2   1705773240  log.error_count      0
3   1705773240    log.row_count    124
4   1705773300  log.error_count      0
..         ...              ...    ...
45  1705774500    log.row_count   1287
46  1705774560  log.error_count      0
47  1705774560    log.row_count   1890
48  1705774620  log.error_count      0
49  1705774620    log.row_count    930

[50 rows x 3 columns],           log_name  total_point_count  p95_threshold
0  log.error_count                 25            0.0
1    log.row_count                 25         1806.0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing dataframes if present
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Ensure timestamps parsed
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Ensure metrics_g (full-series P95 per (cmdb_id,kpi_name)) exists; otherwise compute
try:
    metrics_g
except NameError:
    metrics_g = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'], sort=False)
        .agg(
            total_point_count=('value', 'size'),
            p95_threshold=('value', lambda s: float(s.quantile(0.95)))
        )
        .reset_index()
    )

# Incident window
start = pd.Timestamp("2024-01-20 17:50:28", tz="UTC")
end = pd.Timestamp("2024-01-20 18:20:28", tz="UTC")

# Filter metric rows to the window
df_metric_win = df_metric[(df_metric['timestamp'] >= start) & (df_metric['timestamp'] <= end)].copy()

# Merge full-series p95 into window rows
df_metric_win = df_metric_win.merge(metrics_g[['cmdb_id','kpi_name','p95_threshold']], on=['cmdb_id','kpi_name'], how='left')

# Determine breaches
df_metric_win['is_breach'] = df_metric_win['value'] > df_metric_win['p95_threshold']

# Aggregate per (cmdb_id, kpi_name) within window
# For earliest/last breach timestamps, compute from the subset where is_breach==True
per_kpi_window = (
    df_metric_win
    .groupby(['cmdb_id','kpi_name'], sort=False)
    .agg(
        points_in_window=('value','size'),
        max_value_in_window=('value','max'),
        breach_count=('is_breach','sum'),
        earliest_breach_ts=('timestamp', lambda s: s[s.index][df_metric_win.loc[s.index,'is_breach']].min() if df_metric_win.loc[s.index,'is_breach'].any() else pd.NaT),
        last_breach_ts=('timestamp', lambda s: s[s.index][df_metric_win.loc[s.index,'is_breach']].max() if df_metric_win.loc[s.index,'is_breach'].any() else pd.NaT)
    )
    .reset_index()
)

# Merge p95 and total_point_count from metrics_g
per_kpi = per_kpi_window.merge(
    metrics_g[['cmdb_id','kpi_name','total_point_count','p95_threshold']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# Convert earliest/last breach timestamps to unix seconds (ints) or NaN
def ts_to_unix_or_nan(ts):
    if pd.isna(ts) or ts is pd.NaT:
        return np.nan
    # ts is a pandas Timestamp with tz
    return int(pd.Timestamp(ts).timestamp())

per_kpi['earliest_breach_timestamp'] = per_kpi['earliest_breach_ts'].apply(ts_to_unix_or_nan)
per_kpi['last_breach_timestamp'] = per_kpi['last_breach_ts'].apply(ts_to_unix_or_nan)

# Compute breach_ratio per KPI: (max_value_in_window - p95)/p95 only if breach_count>0 and p95!=0
def compute_breach_ratio(row):
    if row['breach_count'] <= 0:
        return np.nan
    p95 = row['p95_threshold']
    if pd.isna(p95) or p95 == 0:
        return np.nan
    return (row['max_value_in_window'] - p95) / p95

per_kpi['breach_ratio'] = per_kpi.apply(compute_breach_ratio, axis=1)

# Compact KPI-level table
per_kpi_compact = per_kpi[[
    'cmdb_id','kpi_name','total_point_count','p95_threshold',
    'points_in_window','breach_count','earliest_breach_timestamp','last_breach_timestamp',
    'max_value_in_window','breach_ratio'
]].copy()

# Aggregate per service (cmdb_id)
svc_agg = (
    per_kpi_compact
    .groupby('cmdb_id', sort=False)
    .agg(
        sum_of_breach_counts=('breach_count','sum'),
        earliest_breach_timestamp=('earliest_breach_timestamp','min'),
        max_breach_ratio=('breach_ratio','max')
    )
    .reset_index()
)

# Find kpi_with_max_breach per service
# Merge per_kpi_compact to identify KPI(s) achieving the max_breach_ratio
svc_with_kpi = svc_agg.merge(per_kpi_compact[['cmdb_id','kpi_name','breach_ratio']], on='cmdb_id', how='left')
svc_with_kpi['kpi_with_max_breach'] = np.where(
    np.isfinite(svc_with_kpi['max_breach_ratio']) & (svc_with_kpi['breach_ratio'] == svc_with_kpi['max_breach_ratio']),
    svc_with_kpi['kpi_name'],
    np.nan
)

# Pick first non-null kpi_with_max_breach per service
svc_kpi_choice = (
    svc_with_kpi.groupby('cmdb_id', sort=False)
    .agg(
        sum_of_breach_counts=('sum_of_breach_counts','first'),
        earliest_breach_timestamp=('earliest_breach_timestamp','first'),
        max_breach_ratio=('max_breach_ratio','first'),
        kpi_with_max_breach=('kpi_with_max_breach', lambda s: next((x for x in s if pd.notna(x)), np.nan))
    )
    .reset_index()
)

# Sort by earliest_breach_timestamp ascending, NaNs last
svc_kpi_choice_sorted = svc_kpi_choice.sort_values('earliest_breach_timestamp', ascending=True, na_position='last').reset_index(drop=True)

# Limit to top 20
service_summary_top20 = svc_kpi_choice_sorted.head(20).copy()

# Format earliest_breach_timestamp to int where present
service_summary_top20['earliest_breach_timestamp'] = service_summary_top20['earliest_breach_timestamp'].apply(lambda x: int(x) if pd.notna(x) else np.nan)
service_summary_top20['max_breach_ratio'] = service_summary_top20['max_breach_ratio'].astype(float)

# Detailed KPI entries for 'payment' and 'orders'
detailed_kpis_payment_orders = per_kpi_compact[per_kpi_compact['cmdb_id'].isin(['payment','orders'])].copy()
detailed_kpis_payment_orders = detailed_kpis_payment_orders[[
    'cmdb_id','kpi_name','earliest_breach_timestamp','breach_count','max_value_in_window','p95_threshold','breach_ratio'
]].sort_values(['cmdb_id','breach_ratio'], ascending=[True, False]).reset_index(drop=True)

# Return compact tables
service_summary_top20, detailed_kpis_payment_orders

```
Out[4]:
```
```python
# Compose a concise plain-English summary using available variables from prior execution:
import pandas as pd
svc = service_summary_top20.set_index('cmdb_id') if 'service_summary_top20' in globals() else pd.DataFrame()
dets = detailed_kpis_payment_orders if 'detailed_kpis_payment_orders' in globals() else pd.DataFrame()

def fmt_ts(epoch):
    try:
        return pd.to_datetime(int(epoch), unit='s', utc=True).strftime("%Y-%%m-%d %H:%M:%S UTC") if not pd.isna(epoch) else "NaN"
    except Exception:
        return "NaN"

# Build summary lines
lines = []
lines.append("Compact per-service summary for incident window 2024-01-20 17:50:28–18:20:28 UTC:")
lines.append("")

# Orders and Payment overview
for name in ['orders', 'payment']:
    if name in svc.index:
        row = svc.loc[name]
        earliest = pd.to_datetime(int(row['earliest_breach_timestamp']), unit='s', utc=True).strftime("%Y-%m-%d %H:%M:%S UTC") if pd.notna(row['earliest_breach_timestamp']) else "NaN"
        lines.append(f"- {name}: sum_of_breach_counts={int(row['sum_of_breach_counts'])}, earliest_breach_timestamp={int(row['earliest_breach_timestamp']) if pd.notna(row['earliest_breach_timestamp']) else 'NaN'} ({earliest}), max_breach_ratio={row['max_breach_ratio']:.6g}, kpi_with_max_breach={row['kpi_with_max_breach']}")
    else:
        lines.append(f"- {name}: no breach data available")

lines.append("")
lines.append("Key details:")
lines.append("- 'orders' has the highest aggregate breach count (14 breaches across its KPIs). Its worst KPI by relative severity is latency-90 (breach_ratio ≈ 0.3535).")
lines.append("- 'payment' has slightly fewer total breaches (12) but shows the largest single-KPI severity: cpu breach_ratio ≈ 0.6725 (largest relative deviation from its P95). Payment also had a notable diskio breach_ratio ≈ 0.6055.")
lines.append("")
lines.append("Detailed KPI confirmation (selected fields):")
# Summarize detailed KPI table for orders and payment (compact)
if not dets.empty:
    for _, r in dets.iterrows():
        earliest = int(r['earliest_breach_timestamp']) if pd.notna(r['earliest_breach_timestamp']) else "NaN"
        earliest_h = pd.to_datetime(int(earliest), unit='s', utc=True).strftime("%Y-%m-%d %H:%M:%S UTC") if earliest != "NaN" else "NaN"
        lines.append(f"- {r['cmdb_id']} / {r['kpi_name']}: earliest={earliest} ({earliest_h}), breach_count={int(r['breach_count'])}, max_in_window={r['max_value_in_window']}, p95={r['p95_threshold']}, breach_ratio={r['breach_ratio']:.6g}")
else:
    lines.append("- No detailed KPI rows available for payment/orders.")

lines.append("")
lines.append("Takeaway:")
lines.append("- Both services were impacted during the window. 'orders' shows the largest total number of KPI breaches, indicating broader impact across metrics. 'payment' exhibits the single most severe KPI breach (cpu) by relative magnitude, which could indicate a concentrated resource spike on payment.cpu.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id  sum_of_breach_counts  earliest_breach_timestamp  max_breach_ratio kpi_with_max_breach
0             payment                    12                 1705773180          0.672510                 cpu
1              orders                    14                 1705773180          0.353516          latency-90
2        queue-master                     5                 1705773180          0.231802              diskio
3           catalogue                    12                 1705773300          0.043519                 cpu
4          session-db                     6                 1705773300          0.051927              diskio
5            rabbitmq                     4                 1705773420          0.024440                 cpu
6            shipping                    11                 1705773420          0.221555          latency-90
7           front-end                    13                 1705773420          0.057562                 cpu
8             user-db                     6                 1705773420          0.023365              diskio
9                user                    10                 1705773420          0.023792                 cpu
10       catalogue-db                     4                 1705773600          0.010811                 cpu
11              carts                    12                 1705773600          0.414878                 cpu
12          orders-db                     6                 1705773720          0.016781                 cpu
13  rabbitmq-exporter                     2                 1705774080          0.166689                 cpu
14           carts-db                     6                 1705774320          0.050926              diskio,     cmdb_id    kpi_name  earliest_breach_timestamp  breach_count  max_value_in_window  p95_threshold  breach_ratio
0    orders  latency-90               1.705774e+09             2         6.308333e-01   4.660701e-01      0.353516
1    orders         cpu               1.705773e+09             2         7.198180e+00   5.870807e+00      0.226097
2    orders      socket               1.705774e+09             2         1.980000e+01   1.888333e+01      0.048544
3    orders       error               1.705774e+09             2         2.131250e+00   2.053909e+00      0.037656
4    orders  latency-50               1.705774e+09             2         4.336559e-02   4.256430e-02      0.018825
5    orders    workload               1.705773e+09             2         2.177667e+00   2.143740e+00      0.015826
6    orders         mem               1.705773e+09             2         3.375960e+08   3.373040e+08      0.000866
7   payment         cpu               1.705773e+09             2         2.532696e-01   1.514308e-01      0.672510
8   payment      diskio               1.705773e+09             1         7.151727e+04   4.454395e+04      0.605544
9   payment  latency-90               1.705774e+09             2         3.613839e-01   2.933564e-01      0.231894
10  payment      socket               1.705774e+09             1         8.733333e+00   8.000000e+00      0.091667
11  payment  latency-50               1.705774e+09             2         3.602261e-03   3.395280e-03      0.060962
12  payment    workload               1.705774e+09             2         2.173250e+00   2.156620e+00      0.007711
13  payment         mem               1.705775e+09             2         7.335936e+06   7.280886e+06      0.007561)```
```